# Notebook 03: tide, features, windows, split (Section 9.1)

Tide fit on all months of 2018-2022 above floor + 0.05 m (D7) with the G9 residual-spectrum check; feature table; windows inside each season for **train and validation only** (the 2024 test windows are built in Notebook 06 after G18); min-max scalers fitted on training cases, target scaler = Pandacan channel scaler (D9). Script: `python scripts/run_phase.py --phase 3`.

**Runs locally:** the raw data are committed in `data/raw/`; the first cell checks them against the manifest and, in a fresh clone, rebuilds any missing generated inputs (no Google Drive needed).

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from pasig.local import prepare
prepare(3)   # raw data from the repo (no Drive); rebuilds missing local inputs once per clone
import pandas as pd
from pasig import phase3
from pasig.gates import require_gates, read_gate
require_gates('G6', 'G7', 'G8')

In [ ]:
out = phase3.run()

## Tide fit and G9

In [ ]:
print(read_gate('G9')['decision'])
pd.read_csv('reports/tables/tide_constituents.csv').round(4)

In [ ]:
{st: [(it['constituents'][-1], round(it['rmse_fit_m'], 4), {p: round(v['ratio'], 1) for p, v in it['check'].items() if isinstance(v, dict)}) for it in e['iterations']] for st, e in out['tide']['evidence'].items()}

In [ ]:
out['tide_check_2023']

![tide](../reports/figures/fig_tide_pandacan_jul2022.png)

## Windows and scalers

In [ ]:
out['summary']

In [ ]:
json.load(open('artifacts/feature_names.json'))['tab'][:12], json.load(open('artifacts/feature_names.json'))['seq']

## Tests (required tests 2 and 4)

In [ ]:
import subprocess
assert subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_windows.py', 'tests/test_leakage.py']) == 0